In [ ]:
import requests
import re
import os
import time
import random
from dotenv import load_dotenv
from modules.get_token import get_token
import logging
load_dotenv()
start_url = os.getenv("PRJ_START_URL")

In [ ]:
query = """
    query {
      __type(name: "workorder") {
        fields(includeDeprecated: false) {
          name
          type {
            name
            kind
          }
        }
      }
    }
    """

headers = {
    "content-type": "application/json"
}
token = get_token()
url = f"{start_url}/graphql?token={token}"
res = requests.post(url, headers=headers, json={"query": query})
print(res.json()['data'])

{'__type': {'fields': [{'name': 'activeInventoryFlag', 'type': {'kind': 'SCALAR', 'name': 'Boolean'}}, {'name': 'actualDollarAmount', 'type': {'kind': 'SCALAR', 'name': 'Float'}}, {'name': 'allocation', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'asAcceptanceReportNumber1', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'asAcceptanceReportNumber2', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'asAcceptanceReportNumber3', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'asAccountabilityNotes', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'asAdditionalChanges', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'asAssemblyFAI', 'type': {'kind': 'SCALAR', 'name': 'Boolean'}}, {'name': 'asBaselinePartNumber', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'asDetailFAI', 'type': {'kind': 'SCALAR', 'name': 'Boolean'}}, {'name': 'asDrawingNumber', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'asDrawingRevLevel', 'typ

In [ ]:


query = """
    query workOrders($filter: WorkOrderFilter, $pageSize: Int, $pageStart: Int, $query: WorkOrderQuery) {
    workOrders(filter: $filter, pageSize: $pageSize, pageStart: $pageStart, query: $query) {
        records {
            projectCode
            workOrderNumber
            scheduledStartDate
            scheduledEndDate
            asAcceptanceReportNumber2
            dueDate
            dateShipped
            status
            customerPlainText
            customerPONumberPlainText
            quantityOrdered
            qtyComplete
            qtyInWIP
            qtyShipped
            qtyNotYetShipped
            quantityQueuedCalculated
            dateShipped
            daysToShip
            hoursCurrentTarget
            hoursTotalSpent
            setupTimeHoursActualLabel
            setupTimeHoursPlannedTarget
            setupTimeHoursPlannedVarianceLabor
            runningTimeHoursActualLabor
            runningTimeHoursPlannedTargetLabor
            runningTimeHoursPlannedVarianceLabor
            standardizedLaborClass
            standardizedLaborRate
            standardizedLaborClass
            standardProfitPerDLH
            standardizedLaborRate
            partPlainText
            partRev
            pfiPrice
            assemblyClass
            btiPrice
            countAsOnTime
            totalCappedWIP
            totalUncappedWIP
            estWODollarAmount
            type
            wipCogsLabor
            wipCogsMaterials
            wipDirectOverhead
            wipIndirectOverhead
            fairMarketValue
        }

        pageSize
        pageStart
        totalRecords
    }
    }
    """


#=========================== Helper =====================================


def to_float(val):
    if val is None:
        return None

    s = str(val).strip()

    if s in ("", " ", "NULL", "N/A", "-", "--"):
        return None

    s = re.sub(r"[^0-9.\-]", "", s)

    if s in ("", "-", "."):
        return None

    try:
        return float(s)
    except ValueError:
        return None


def to_int(val):
    try:
        if val in (None, "", " ", "NULL", "N/A", "-", "--"):
            return None
        val = str(val).replace(",", "").strip()
        if val == "":
            return None
        return int(float(val))
    except Exception:
        return None






#=========================================================================


def fetch_page(token, page_start, page_size, session):
    """Fetch work orders page"""
    logging.info("Fetching all work orders")
    url = f"{start_url}/graphql?token={token}"

    payload = {
        "query": query,
        "variables": {
            "filter": {"activeInventoryFlag": True},
            "pageSize": page_size,
            "pageStart": page_start,
            "query": {
                "lastModifiedTime": {
                    "greaterThanOrEqual": "2024-01-01T00:00:00Z"
                }
            }
        }
    }

    for attempt in range(3):
        try:
            res = session.post(url, headers=headers, json=payload)
            res.raise_for_status()
            print(res.json())
            if res.status_code == 401:
                token = get_token()
                continue
            res.raise_for_status()
            data = res.json()["data"]["workOrders"]
            print(data)
            return {
                "records": data["records"],
                "total": data["totalRecords"]
            }

        except requests.exceptions.ReadTimeout:
            logging.warning(f"Timeout on page {page_start}, retry {attempt+1}/3")
            time.sleep(random.uniform(3, 7))

    logging.error(f"Failed after retries on page {page_start}")
    return {"records": [], "total": 0}  

    

In [ ]:
if __name__ == "__main__":
    session = requests
    token = get_token()
    print(token)
    if token:
        fetch_page(token, 0, 100, session)